# ICR TC1 Human Baseline Work Notebook

This notebook authors the `tc1_from_scratch` human baseline directly against the current action bank.

Mapped bank actions in this notebook:
- `CA-000002` drop `Id`
- `CA-000006` encode `EJ`
- `CA-000004` median-impute sparse numeric biomarkers
- `CA-000008` join `greeks.csv`
- `CA-000010` parse `Epsilon`
- `CA-000012` standard-scale numeric biomarkers
- `CA-000014` selective log transform for skewed biomarkers


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

TASK_DIR = Path('..')
DATA_DIR = TASK_DIR / 'data'

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
greeks = pd.read_csv(DATA_DIR / 'greeks.csv')

print({'train': train.shape, 'test': test.shape, 'greeks': greeks.shape})


## Inspect Sparse Columns And Header Quirk

The current ontology cannot represent rename-columns directly, so header cleanup is treated as a workflow assumption rather than a bank action.


In [ ]:
train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()

sparse_cols = ['BQ', 'EL', 'CC', 'CB', 'FS', 'DU', 'FC', 'FL', 'GL']
missing_summary = train[sparse_cols].isna().sum().sort_values(ascending=False)
print(missing_summary.to_dict())


## Build Core Modeling Frame

This section covers `CA-000002`, `CA-000006`, and `CA-000004`.


In [ ]:
core_train = train.copy()
core_test = test.copy()

# CA-000006: encode EJ as a binary category.
ej_mapping = {'A': 0, 'B': 1}
core_train['EJ'] = core_train['EJ'].map(ej_mapping).astype('int64')
core_test['EJ'] = core_test['EJ'].map(ej_mapping).astype('int64')

# CA-000004: median-impute the sparse numeric subset.
median_imputer = SimpleImputer(strategy='median')
core_train[sparse_cols] = median_imputer.fit_transform(core_train[sparse_cols])
core_test[sparse_cols] = median_imputer.transform(core_test[sparse_cols])

# CA-000002: remove Id from the model-side feature frame.
model_train = core_train.drop(columns=['Id', 'Class'])
model_test = core_test.drop(columns=['Id'])

print(model_train[['EJ'] + sparse_cols].head())


## Join Official Greeks Metadata

This section covers `CA-000008` and `CA-000010`. It is a validation-side branch, not part of the minimal primary submission spine.


In [ ]:
greeks_work = greeks.copy()

# CA-000010: parse Epsilon with coercion because Unknown appears in the official file.
greeks_work['Epsilon'] = pd.to_datetime(greeks_work['Epsilon'], errors='coerce', format='%m/%d/%Y')

# CA-000008: left-join the official greeks metadata onto train rows.
train_with_greeks = core_train[['Id', 'Class']].merge(
    greeks_work[['Id', 'Alpha', 'Beta', 'Gamma', 'Delta', 'Epsilon']],
    on='Id',
    how='left'
)

train_with_greeks['Epsilon_ordinal'] = train_with_greeks['Epsilon'].map(
    lambda value: value.toordinal() if pd.notnull(value) else np.nan
)
print(train_with_greeks[['Alpha', 'Beta', 'Gamma', 'Delta', 'Epsilon', 'Epsilon_ordinal']].head())


## Selective Log Transform For Skewed Biomarkers

This section covers `CA-000014` using a column subset that appears repeatedly in the selected notebook corpus.


In [ ]:
log_cols = ['AX', 'BP', 'CC', 'CF', 'CR', 'CS', 'CU', 'DE', 'EE', 'EG', 'FC', 'FE', 'FI', 'GB', 'GF']
log_branch_train = model_train.copy()
log_branch_test = model_test.copy()

log_branch_train[log_cols] = np.log1p(log_branch_train[log_cols])
log_branch_test[log_cols] = np.log1p(log_branch_test[log_cols])

print(log_branch_train[log_cols].head())


## Standard-Scale Numeric Biomarkers

This section covers `CA-000012` on top of the selective-log branch.


In [ ]:
scaled_train = log_branch_train.copy()
scaled_test = log_branch_test.copy()

numeric_cols = [column for column in scaled_train.columns if column != 'EJ']
scaler = StandardScaler()
scaled_train[numeric_cols] = scaler.fit_transform(scaled_train[numeric_cols])
scaled_test[numeric_cols] = scaler.transform(scaled_test[numeric_cols])

print(scaled_train[numeric_cols[:5]].head())


## Final Baseline Selection

This notebook intentionally chooses one representative branch per need rather than selecting every compatible bank row.


In [ ]:
selected_action_ids = [
    'CA-000002',
    'CA-000006',
    'CA-000004',
    'CA-000008',
    'CA-000010',
    'CA-000012',
    'CA-000014',
]

selected_action_ids
